# Lección 14.3 — Q-learning y redes Q profundas: la chuleta del agente

> **Módulo 14 · Aprendizaje por refuerzo (introducción conceptual)** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 14.1 y 14.2; Lección 3.3 (descenso de gradiente) y Lección 9.1 (la neurona) para la parte de redes

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es el **valor Q** de una acción y cómo una **tabla Q** define una política.
- Describir con palabras cómo **aprende** Q-learning (la regla de actualización) y aplicarla a mano.
- Predecir el efecto de los hiperparámetros **α (tasa de aprendizaje)**, **γ** y **ε** en Q-learning.
- Explicar por qué las tablas no escalan y qué cambia en una **red Q profunda (DQN)**: red neuronal, memoria de repetición y red objetivo.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: la chuleta del taxista 🚕

Un taxista novato no conoce la ciudad. Uno veterano tiene en la cabeza una especie de **chuleta**:

> *"Si estoy en **esta esquina** y doblo a la **derecha**, llego al aeropuerto en unos **12 minutos**. Si sigo **recto**, en **20**."*

Para cada **situación** (esquina) y cada **acción** (girar, seguir), sabe **qué tan buena** es esa decisión pensando en **todo el viaje que queda**, no solo en la siguiente cuadra. Con esa chuleta, conducir es fácil: en cada esquina, elige la acción con el mejor número.

**Q-learning aprende exactamente esa chuleta.** El número se llama **valor Q** (*Q-value*): Q(estado, acción) = el retorno que espero conseguir si en este estado hago esta acción y después sigo actuando bien.

### ✋ Ejercicio 1 — Llena la chuleta a mano
Volvamos al pasillo del ratón de la Lección 14.1 (−1 por paso, +10 al llegar al queso en la casilla 6, sin descuento: γ = 1).

```
casilla:   0    1    2    3    4    5    6
          😾   ⬜   ⬜   ⬜   ⬜   ⬜   🧀
```

¿Cuánto vale Q(casilla, ➡️) para las casillas **5, 4 y 3**, si después de ese paso el ratón sigue yendo a la derecha?

✍️ *Q(5, ➡️) = ___ · Q(4, ➡️) = ___ · Q(3, ➡️) = ___*

<details><summary><b>👀 Ver respuesta</b></summary>

- Q(5, ➡️) = **10** (llega directo al queso).
- Q(4, ➡️) = −1 + 10 = **9**.
- Q(3, ➡️) = −1 − 1 + 10 = **8** (justo el retorno que calculaste en la Lección 14.1).

¿Ves el patrón? **El valor de una casilla se construye con el valor de la siguiente**: Q(4) = recompensa del paso + Q(5). Esta idea, que el valor de hoy se apoya en el de mañana, es el corazón de Q-learning (y se llama **ecuación de Bellman**).
</details>

---
## 2. 📖 El concepto: la tabla Q

Si hay pocos estados y pocas acciones, la chuleta cabe en una **tabla**: una fila por estado, una columna por acción.

| Estado | ⬆️ | ➡️ | ⬇️ | ⬅️ |
|---|---|---|---|---|
| casilla (0,0) | 2.1 | **5.3** | 4.0 | 2.1 |
| casilla (0,1) | 3.2 | **5.9** | 1.1 | 4.2 |
| … | … | … | … | … |

- **Política del agente:** en cada estado, elegir la acción con el **Q más alto** (en negrita).
- **Aprender** = ir rellenando y corrigiendo esos números con la experiencia.

### 📖 Nuestro laberinto
Un robot 🤖 debe llegar al diamante 💎 en este laberinto de 5 × 5:

```
🟩 ⬜ ⬜ ⬜ ⬜        🟩 salida        🧱 pared (chocar = quedarse quieto)
⬜ 🧱 🧱 ⬜ 🔥        🔥 lava: −10 y fin del episodio
⬜ ⬜ 🧱 ⬜ ⬜        💎 diamante: +10 y fin del episodio
🧱 ⬜ ⬜ ⬜ 🧱        ⬜ cualquier otro paso: −1
⬜ ⬜ 🧱 ⬜ 💎
```

La tabla Q tiene 25 casillas × 4 acciones = **100 números**. Al principio todos valen 0: el robot no sabe nada.

In [ ]:
MAPA = ["S....",
        ".##.X",
        "..#..",
        "#...#",
        "..#.G"]
ACCIONES = ["⬆️", "➡️", "⬇️", "⬅️"]
MOVIMIENTOS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
ICONOS = {"S": "🟩", ".": "⬜", "#": "🧱", "X": "🔥", "G": "💎"}

class Laberinto:
    def __init__(self, mapa, costo_paso=-1, premio_meta=10, castigo_lava=-10, resbalon=0.0, semilla=0):
        self.mapa, self.F, self.C = mapa, len(mapa), len(mapa[0])
        self.inicio = next((f, c) for f in range(self.F) for c in range(self.C) if mapa[f][c] == "S")
        self.costo_paso, self.premio_meta, self.castigo_lava = costo_paso, premio_meta, castigo_lava
        self.resbalon = resbalon                       # probabilidad de resbalar (acción al azar)
        self.rng = np.random.default_rng(semilla)

    def reiniciar(self):
        self.pos = self.inicio
        return self.pos

    def paso(self, a):
        if self.rng.random() < self.resbalon:          # ¡se resbaló! hace una acción al azar
            a = self.rng.integers(4)
        f, c = self.pos[0] + MOVIMIENTOS[a][0], self.pos[1] + MOVIMIENTOS[a][1]
        if 0 <= f < self.F and 0 <= c < self.C and self.mapa[f][c] != "#":
            self.pos = (f, c)                          # si choca con pared o borde, se queda quieto
        celda = self.mapa[self.pos[0]][self.pos[1]]
        if celda == "G": return self.pos, self.premio_meta, True
        if celda == "X": return self.pos, self.castigo_lava, True
        return self.pos, self.costo_paso, False

def dibujar(mapa, Q=None, camino=()):
    """Dibuja el mapa. Con Q, pone en cada casilla libre la flecha de la mejor acción."""
    for f, fila in enumerate(mapa):
        linea = ""
        for c, celda in enumerate(fila):
            if (f, c) in camino and celda in "S.":
                linea += "🤖"
            elif Q is not None and celda in "S.":
                linea += ACCIONES[int(np.argmax(Q[f, c]))]
            else:
                linea += ICONOS[celda]
        print(linea)

dibujar(MAPA)

### 🔮 Ejercicio 2 — El camino ideal
Cuenta en el mapa: ¿cuántos pasos tiene el camino **más corto** del 🟩 al 💎? ¿Hay más de uno? ¿Qué retorno obtendría el robot con ese camino (−1 por paso normal, +10 al llegar)?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

El camino más corto tiene **8 pasos** y hay **dos**: por arriba (➡️➡️➡️ y luego ⬇️⬇️⬇️⬇️➡️, pasando junto a la lava) o por la izquierda (⬇️⬇️➡️⬇️➡️➡️⬇️➡️, más o menos). Retorno: 7 pasos normales (−7) + 10 = **3**.
</details>

---
## 3. 📖 Cómo aprende: corregir la chuleta paso a paso

El robot explora con **ε-greedy** (Lección 14.2). Después de **cada paso** mira lo que pasó y corrige **una sola casilla** de la tabla:

```
            lo que acabo de vivir                  lo que creía
         ┌───────────────────────────┐            ┌────────┐
objetivo = recompensa + γ · (mejor Q del estado nuevo)
error    = objetivo − Q(estado, acción)
Q(estado, acción) ← Q(estado, acción) + α · error
```

En palabras: *"Creía que esta acción valía Q. Acabo de ver que me dio una recompensa y me dejó en un lugar que (según mi propia chuleta) vale tanto. Muevo mi estimación **un poquito** (α) hacia esa nueva evidencia."*

- **α** (alfa, tasa de aprendizaje): qué tanto corrijo en cada paso. Es la misma idea que la tasa de aprendizaje del descenso de gradiente (Lección 3.4).
- **γ**: el factor de descuento de la Lección 14.2.

> 💡 Fíjate en el truco: el objetivo usa **la propia tabla** (la "mejor Q del estado nuevo"). El agente aprende de sus propias estimaciones, que a su vez van mejorando. Se llama ***bootstrapping*** (como levantarse tirando de los cordones de sus propias botas).

### ✋ Ejercicio 3 — Una actualización a mano
El robot está en una casilla donde cree que Q(casilla, ➡️) = **2**. Va a la derecha, recibe **−1** y llega a una casilla cuya mejor acción vale **5**. Con **γ = 0.9** y **α = 0.5**:
1. ¿Cuál es el objetivo?
2. ¿Cuál es el error?
3. ¿Cuál es el nuevo Q(casilla, ➡️)?

✍️ *Mis cálculos:*

In [ ]:
def actualizar(q_viejo, recompensa, mejor_q_siguiente, alfa, gamma, terminado=False):
    objetivo = recompensa + (0 if terminado else gamma * mejor_q_siguiente)
    return q_viejo + alfa * (objetivo - q_viejo)

print(actualizar(q_viejo=2, recompensa=-1, mejor_q_siguiente=5, alfa=0.5, gamma=0.9))   # ✏️ cambia α

<details><summary><b>👀 Ver respuesta</b></summary>

1. Objetivo = −1 + 0.9 × 5 = **3.5**.
2. Error = 3.5 − 2 = **1.5** (la acción resultó mejor de lo que creía).
3. Nuevo Q = 2 + 0.5 × 1.5 = **2.75**.

Con α = 1 saltaría directo a 3.5 (olvida todo lo anterior); con α = 0.1 apenas se movería a 2.15.
</details>

▶️ Ahora sí, el algoritmo completo. Son pocas líneas: repetir episodios, elegir con ε-greedy, actualizar.

In [ ]:
def q_learning(lab, episodios=500, alfa=0.1, gamma=0.9, epsilon=0.1, max_pasos=100, semilla=0):
    rng = np.random.default_rng(semilla)
    Q = np.zeros((lab.F, lab.C, 4))                # la chuleta empieza en blanco
    pasos_por_episodio = []
    for _ in range(episodios):
        s = lab.reiniciar()
        for t in range(1, max_pasos + 1):
            if rng.random() < epsilon:                                 # explorar
                a = rng.integers(4)
            else:                                                      # explotar (empates al azar)
                a = rng.choice(np.flatnonzero(Q[s] == Q[s].max()))
            s2, r, terminado = lab.paso(a)
            objetivo = r + (0 if terminado else gamma * Q[s2].max())
            Q[s][a] += alfa * (objetivo - Q[s][a])                     # ← la regla de Q-learning
            s = s2
            if terminado:
                break
        pasos_por_episodio.append(t)
    return Q, pasos_por_episodio

def seguir_politica(mapa, Q, max_pasos=30):
    """Sigue la mejor acción de la tabla (sin explorar ni resbalar). Devuelve el camino y si llegó."""
    lab = Laberinto(mapa)
    s, camino = lab.reiniciar(), [lab.inicio]
    for _ in range(max_pasos):
        s, r, terminado = lab.paso(int(np.argmax(Q[s])))
        camino.append(s)
        if terminado:
            return camino, "💎 llegó a la meta" if r > 0 else "🔥 cayó en la lava"
    return camino, "😵 se quedó dando vueltas"

### 🔮 Ejercicio 4 — La curva de aprendizaje
Graficaremos cuántos **pasos** tarda el robot en terminar cada episodio.
1. ¿Cómo será la curva al principio: muchos o pocos pasos?
2. ¿A qué número debería acercarse al final? (Pista: Ejercicio 2… pero recuerda que con ε = 0.1 sigue explorando un 10 % del tiempo.)

✍️ *Mis predicciones:*

In [ ]:
lab = Laberinto(MAPA)
Q, pasos = q_learning(lab, episodios=500)

plt.plot(pasos, alpha=.4, label="pasos en cada episodio")
plt.plot(np.convolve(pasos, np.ones(20) / 20, mode="valid"), lw=2, label="media móvil (20 episodios)")
plt.axhline(8, ls="--", c="gray", label="camino óptimo (8)")
plt.xlabel("Episodio"); plt.ylabel("Pasos hasta terminar"); plt.ylim(0, 100); plt.legend(); plt.grid(alpha=.3); plt.show()

print(f"Pasos promedio: primeros 20 episodios = {np.mean(pasos[:20]):.1f} · últimos 100 = {np.mean(pasos[-100:]):.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Al principio, **muchos pasos** (decenas, a veces el máximo de 100): la tabla está en blanco y el robot deambula casi al azar.
2. Al final se acerca a **8** pero queda un poco por encima (≈ 9 de media): el 10 % de acciones al azar de vez en cuando lo desvía.

La caída es rápida: en unas pocas decenas de episodios ya encontró un buen camino.
</details>

▶️ Veamos **la chuleta aprendida** como un mapa de flechas (la mejor acción en cada casilla) y el camino que sigue el robot:

In [ ]:
print("Política aprendida (mejor acción en cada casilla):")
dibujar(MAPA, Q)
camino, final = seguir_politica(MAPA, Q)
print(f"\nCamino del robot ({len(camino) - 1} pasos, {final}):")
dibujar(MAPA, camino=camino)
print("\nValores Q de la salida:", dict(zip(ACCIONES, Q[0, 0].round(2))))

### ✋ Ejercicio 5 — Lee la chuleta
1. ¿Qué camino eligió el robot: el de arriba o el de la izquierda?
2. Mira las flechas de las casillas **por las que el robot casi nunca pasa** (por ejemplo, abajo a la izquierda). ¿Tienen sentido todas?
3. Q(salida, ➡️) debería acercarse al retorno **descontado** del camino óptimo: −1 − 0.9 − 0.81 − … − 0.9⁶ + 0.9⁷·10 ≈ **−0.44**. ¿Se parece?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con esta semilla, el robot sale por la **derecha** (camino de arriba, junto a la lava), en **8 pasos**.
2. En las casillas poco visitadas algunas flechas pueden ser raras (apuntar a una pared, por ejemplo): **el agente solo aprende bien lo que explora**. No importa para el camino principal, pero si el robot empezara en otro lugar, podría perderse.
3. Q(salida, ➡️) ≈ **−0.43**: ¡casi exacto! Nadie le dio esa cuenta al robot; la obtuvo solo, a base de experiencia. (Ojo: en casillas poco visitadas los valores todavía no son tan precisos; lo que más importa es que **ordenen bien** las acciones.)

> 📝 Las acciones ⬆️ y ⬅️ en la salida chocan contra el borde: el robot se queda quieto y paga −1. Por eso valen menos.
</details>

---
## 4. 📖 ¿Qué pasa si el mundo es enorme? De la tabla a la red

Nuestra tabla tiene 100 números. Pero piensa en otros problemas:

| Problema | Estados aproximados | ¿Cabe en una tabla? |
|---|---|---|
| Nuestro laberinto | 25 | ✅ |
| Gato (tres en raya) | ~5,500 | ✅ |
| Ajedrez | ~10⁴⁴ | ❌ más que granos de arena en la Tierra |
| Videojuego de Atari visto como imagen (84×84 píxeles, 256 tonos) | 256^(84·84) | ❌❌❌ |

### 🔮 Ejercicio 6 — ¿Cuánto mide la tabla?
Un robot de bodega ve su posición en una cuadrícula de **100 × 100**, su nivel de batería en **10** niveles y si lleva o no un paquete. Tiene **6** acciones. ¿Cuántos números tendría su tabla Q? Y aunque cupiera en memoria, ¿cuál es el **otro** problema?

✍️ *Mi cálculo:*

<details><summary><b>👀 Ver respuesta</b></summary>

100 × 100 × 10 × 2 = 200,000 estados × 6 acciones = **1,200,000 números**. Cabe en memoria, pero el problema real es otro: **el agente tendría que visitar cada casilla de la tabla muchas veces** para aprender su valor. Y lo que aprenda en la posición (40, 50) **no le sirve de nada** en la (40, 51), aunque sean casi iguales: una tabla **no generaliza**.
</details>

### 📖 La idea de DQN (*Deep Q-Network*, DeepMind 2015)
En vez de una tabla, usa una **red neuronal** (Módulo 9) que recibe el estado y **predice** los valores Q de todas las acciones:

```
  TABLA Q                          RED Q (DQN)
  estado ─► buscar fila ─► Q       estado ─► [ red neuronal ] ─► Q(⬆️) Q(➡️) Q(⬇️) Q(⬅️)
  (memoriza cada casilla)          (generaliza: estados parecidos → valores parecidos)
```

La regla de aprendizaje es la misma: el "objetivo" `r + γ·max Q(siguiente)` se convierte en la **etiqueta** de un problema de **regresión**, y la red se entrena con descenso de gradiente (Lección 3.3) para acercarse a ella.

Pero entrenar así es **muy inestable**: la red persigue un objetivo que ella misma va moviendo. DeepMind necesitó dos trucos clave para que funcionara en 49 juegos de Atari:

| Truco | Qué hace | Analogía |
|---|---|---|
| **Memoria de repetición** (*experience replay*) | Guarda miles de experiencias (estado, acción, recompensa, estado siguiente) y entrena con **lotes al azar** de esa memoria | Estudiar repasando tu cuaderno entero, no solo la última página |
| **Red objetivo** (*target network*) | Calcula los objetivos con una **copia congelada** de la red, que se actualiza cada cierto tiempo | No mover la portería mientras intentas meter el gol |

▶️ Vamos a construir un **mini-DQN por lotes** con esas dos ideas. Como aquí no tenemos PyTorch, usaremos como aproximador un **bosque de árboles** (`ExtraTreesRegressor`, primo del Random Forest del Módulo 6) en lugar de una red: la lógica es idéntica y corre en segundos.

**Paso A — La memoria de repetición:** el robot da 1,500 pasos **totalmente al azar**, empezando en casillas al azar, y guarda todo lo que vive.

In [ ]:
rng = np.random.default_rng(0)
lab = Laberinto(MAPA)
libres = [(f, c) for f in range(lab.F) for c in range(lab.C) if MAPA[f][c] in "S."]

memoria = []                                    # (estado, acción, recompensa, estado_siguiente, terminado)
for _ in range(1500):
    lab.pos = libres[rng.integers(len(libres))]
    s, a = lab.pos, rng.integers(4)
    s2, r, terminado = lab.paso(a)
    memoria.append((s, a, r, s2, terminado))

print(f"Experiencias en memoria: {len(memoria)}")
print("Ejemplo:", memoria[0])

**Paso B — Entrenar el aproximador con objetivos de una copia congelada.** En cada iteración:
1. la versión **anterior** del modelo (la "red objetivo", congelada) calcula los objetivos `r + γ·max Q(siguiente)`;
2. se entrena un modelo **nuevo** para predecir esos objetivos a partir de *(fila, columna, acción)*.

### 🔮 Ejercicio 7 — ¿Puede aprender solo de experiencias al azar?
La memoria se llenó con un robot que se movía **completamente al azar**: nunca intentó llegar al diamante. ¿Crees que el modelo aprenderá aun así una política que **sí** llega al diamante? ¿Y si solo hacemos **3 iteraciones**?

✍️ *Mi predicción:*

In [ ]:
ITERACIONES = 15     # ✏️ prueba 3 y luego 15
GAMMA = 0.9

S  = np.array([m[0] for m in memoria]); A = np.array([m[1] for m in memoria])
R  = np.array([m[2] for m in memoria]); S2 = np.array([m[3] for m in memoria])
FIN = np.array([m[4] for m in memoria])

def entradas(estados, acciones):                 # la "foto" que ve el modelo: fila, columna y acción
    return np.column_stack([estados, acciones])

modelo = None
for it in range(ITERACIONES):
    if modelo is None:
        objetivo = R                             # primera vuelta: solo la recompensa inmediata
    else:                                        # 'modelo' es la copia congelada de la vuelta anterior
        q_sig = np.column_stack([modelo.predict(entradas(S2, np.full(len(S2), a))) for a in range(4)])
        objetivo = R + GAMMA * (1 - FIN) * q_sig.max(axis=1)
    modelo = ExtraTreesRegressor(n_estimators=50, random_state=0).fit(entradas(S, A), objetivo)

# Convertimos el modelo en una "tabla" para poder dibujarlo con las mismas funciones
Q_aprox = np.zeros((lab.F, lab.C, 4))
for f, c in libres:
    Q_aprox[f, c] = modelo.predict(entradas(np.array([[f, c]] * 4), np.arange(4)))

dibujar(MAPA, Q_aprox)
camino, final = seguir_politica(MAPA, Q_aprox)
print(f"\nCon {ITERACIONES} iteraciones: {len(camino) - 1} pasos, {final}")

<details><summary><b>👀 Ver respuesta</b></summary>

**Sí aprende**, y es una propiedad muy importante de Q-learning: puede aprender la mejor política **observando experiencias de otra política** (aquí, una totalmente al azar). Se dice que es ***off-policy***. Gracias a eso DQN puede reutilizar experiencias viejas de su memoria.

- Con **3 iteraciones** el robot **no** llega: la información del diamante solo ha "viajado" 3 casillas hacia atrás (cada iteración la propaga un paso). Desde la salida, a 8 pasos, todas las acciones parecen igual de malas.
- Con **15 iteraciones** la información ya recorrió todo el laberinto y el robot llega en **8 pasos**.

Es la misma propagación "de la meta hacia atrás" del Ejercicio 1, ahora hecha con un modelo de ML.
</details>

### 📖 Para Colab: un DQN de verdad con PyTorch
Si quieres ver un DQN completo con red neuronal, abre un notebook en **Google Colab** y ejecuta esto (no corre aquí porque requiere instalar `stable-baselines3` y PyTorch). Entrena un agente que aprende a equilibrar un palo sobre un carrito (*CartPole*):

```python
# ⚠️ Para Colab
!pip install stable-baselines3 gymnasium
import gymnasium as gym
from stable_baselines3 import DQN

entorno = gym.make("CartPole-v1")
agente = DQN("MlpPolicy", entorno,
             learning_rate=1e-3,            # α de la red
             buffer_size=50_000,            # tamaño de la memoria de repetición
             batch_size=64,                 # experiencias por lote
             gamma=0.99,
             exploration_fraction=0.2,      # fracción del entrenamiento en que ε decae…
             exploration_final_eps=0.05,    # …hasta este valor
             target_update_interval=500,    # cada cuántos pasos se copia la red objetivo
             verbose=1)
agente.learn(total_timesteps=50_000)
```

---
## 5. 📖 Hiperparámetros: las perillas de Q-learning

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `alfa` (α) | Cuánto corrige Q en cada paso | Aprende rápido pero "nervioso": oscila con el ruido | Estable pero lento | 0.1 (tabla) · 1e-4 a 1e-3 (red) |
| `gamma` (γ) | Qué tan lejos mira | Planea a largo plazo, aprende más lento | Miope: puede ignorar la meta | 0.9–0.99 |
| `epsilon` (ε) | Exploración (Lección 14.2) | Descubre más, rinde menos mientras aprende | Puede no descubrir el buen camino | 1.0 que decae a 0.01–0.1 |
| `episodios` | Cuánta experiencia | Mejor tabla (con rendimientos decrecientes) | Tabla a medio llenar | Hasta que la curva se aplane |
| `buffer_size` (DQN) | Tamaño de la memoria de repetición | Experiencias más variadas, pero más viejas | Más reciente, menos variada (inestable) | 10⁴–10⁶ |
| `target_update_interval` (DQN) | Cada cuánto se copia la red objetivo | Más estable, aprende más lento | Más rápido, más inestable | 500–10,000 pasos |

### ✏️ Ejercicio 8 — α en un piso resbaloso
Hagamos el mundo más difícil: con `resbalon=0.2` el robot hace una acción **al azar** el 20 % de las veces (como un piso mojado). Ahora la misma acción no siempre da el mismo resultado: hay **ruido**.

Compararemos α = 0.01, 0.1, 0.5 y 1.0 promediando 20 robots por valor. ¿Cuál crees que terminará con **menos pasos** por episodio al final? ¿Cuál aprenderá más lento al principio?

✍️ *Mis predicciones:*

In [ ]:
ALFAS = [0.01, 0.1, 0.5, 1.0]     # ✏️

for alfa in ALFAS:
    curvas = [q_learning(Laberinto(MAPA, resbalon=0.2, semilla=s), episodios=300, alfa=alfa, semilla=s)[1]
              for s in range(20)]
    curvas = np.array(curvas)
    print(f"α = {alfa:<5} → pasos promedio: primeros 50 episodios = {curvas[:, :50].mean():5.1f} · "
          f"últimos 50 = {curvas[:, -50:].mean():5.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **α = 0.01:** aprende **lentísimo** (muchos pasos al principio y todavía bastantes al final): corrige tan poco que 300 episodios no le alcanzan.
- **α = 0.1 y 0.5:** los mejores al final (≈ 11–12 pasos; recuerda que resbalar alarga el camino).
- **α = 1.0:** aprende rápido al principio pero termina **peor**: cada resbalón le hace "olvidar" todo lo anterior y reescribir el valor con un solo dato ruidoso.

Es la misma historia que la tasa de aprendizaje del descenso de gradiente (Lección 3.4): **demasiado pequeña es lenta; demasiado grande, inestable.**
</details>

### 🔮 Ejercicio 9 — Un robot miope
Volvemos al piso normal (sin resbalar). Entrenamos con γ = **0**, **0.3** y **0.9**.
- Con γ = 0 el robot solo ve la recompensa inmediata.
- Con γ = 0.3 el diamante, a 8 pasos, vale 0.3⁷ × 10 ≈ 0.002: prácticamente nada.

¿Cuál de los tres llegará al diamante?

✍️ *Mi predicción:*

In [ ]:
for gamma in [0.0, 0.3, 0.9]:       # ✏️
    Q_g, _ = q_learning(Laberinto(MAPA), episodios=500, gamma=gamma)
    camino, final = seguir_politica(MAPA, Q_g)
    print(f"γ = {gamma:<5} → {final} ({len(camino) - 1} pasos)")

<details><summary><b>👀 Ver respuesta</b></summary>

- **γ = 0:** se queda **dando vueltas**. Para él, casi cualquier paso vale −1 (la recompensa inmediata) y le da igual adónde ir: el diamante no existe hasta que está a un paso.
- **γ = 0.3:** ¡**sí llega**, en 8 pasos! Sorpresa: aunque el diamante casi no cuenta, el costo de −1 por paso hace que los caminos más cortos valgan *un poquito* más. En un mundo sin ruido esas diferencias minúsculas bastan para ordenar las acciones, pero son **frágiles**: cualquier ruido (resbalones, pocas visitas) las puede tapar.
- **γ = 0.9:** llega en 8 pasos con valores claramente distintos para cada acción.

Recuerda la regla de la Lección 14.2: el horizonte es ≈ 1/(1 − γ). Con 0.3 son menos de 2 pasos de "visión".
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La curva de pasos no baja nunca | La recompensa casi nunca se alcanza; ε muy bajo; γ muy bajo | Más exploración, recompensas intermedias, subir γ |
| La curva baja y luego **oscila** sin estabilizarse | α demasiado alto con un entorno ruidoso | Bajar α o hacerlo decaer |
| El agente funciona desde la salida, pero se pierde si empieza en otro lugar | Solo aprendió bien los estados que visitó | Empezar episodios en estados al azar, más exploración |
| (DQN) La pérdida sube de golpe o los Q crecen sin control | Inestabilidad: objetivos que se mueven, recompensas grandes | Red objetivo, bajar `learning_rate`, recortar recompensas a [−1, 1] |
| (DQN) Olvida lo que ya sabía | Memoria demasiado pequeña: solo recuerda lo reciente | Subir `buffer_size` |
| Los valores Q son demasiado optimistas | El `max` sobre estimaciones ruidosas sobreestima (*maximization bias*) | Double Q-learning / Double DQN |

### 🔮 Ejercicio 10 — Diagnóstico
Un compañero entrena un robot con Q-learning siempre desde la casilla de salida. Funciona perfecto. Luego, en la demostración, lo colocan en la esquina inferior izquierda del laberinto y el robot **choca una y otra vez contra una pared**. ¿Qué pasó y cómo lo arreglarías?

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

La tabla solo tiene buenos valores en los estados que el robot **visitó durante el entrenamiento**. La esquina inferior izquierda casi nunca se visitaba, así que sus Q siguen cerca de 0 o mal estimados, y la flecha apunta a cualquier sitio. Es la fila 3 de la tabla: **arreglo** → entrenar empezando desde casillas al azar (como hicimos en la memoria del mini-DQN) o explorar más. Es el equivalente RL de "el modelo no generaliza fuera de los datos que vio" (Lección 4.4).
</details>

---
## 7. 📋 Ficha del modelo

| | Q-learning (tabla) | DQN (red Q profunda) |
|---|---|---|
| **Intuición** | Una chuleta: qué tan buena es cada acción en cada estado | La chuleta la "calcula" una red neuronal |
| **Cómo aprende** | Corrige Q(s, a) hacia `r + γ·max Q(s')` un poquito (α) en cada paso | Igual, como un problema de regresión, con memoria de repetición y red objetivo |
| **Hiperparámetros clave** | α, γ, ε, episodios | `learning_rate`, γ, ε (decaimiento), `buffer_size`, `batch_size`, `target_update_interval` |
| **✅ Fortalezas** | Simple, interpretable, garantizado en problemas pequeños · Aprende de experiencias de otra política (*off-policy*) | Escala a estados enormes (imágenes) · Generaliza |
| **❌ Debilidades** | No escala ni generaliza | Inestable, sensible a hiperparámetros, necesita millones de pasos |
| **Úsalo cuando…** | Pocos estados y acciones discretas | Estados grandes (imágenes, sensores) y **pocas acciones discretas** |
| **Evítalo cuando…** | Hay millones de estados | Las acciones son **continuas** (girar un volante 23.7°): usa gradientes de política (Lección 14.4) |

## 8. 🧠 Resumen en 5 puntos
1. **Q(estado, acción)** es el retorno que esperas si haces esa acción y luego sigues actuando bien; la política es "elegir el Q más alto".
2. **Q-learning** corrige la tabla en cada paso hacia `recompensa + γ · mejor Q siguiente`, con un paso de tamaño **α**.
3. El valor se propaga **desde la meta hacia atrás**, episodio a episodio.
4. Es ***off-policy***: puede aprender la mejor política a partir de experiencias de cualquier otra (incluso al azar).
5. Cuando los estados son demasiados, **DQN** reemplaza la tabla por una red neuronal y se estabiliza con **memoria de repetición** y **red objetivo**.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** En Q-learning, ¿qué significa que α sea muy alto (cercano a 1)?
- a) El agente explora más · b) Cada experiencia nueva reemplaza casi por completo la estimación anterior · c) El agente mira más lejos · d) La tabla se hace más grande

**2.** Q(s, ⬆️) = 4, Q(s, ➡️) = 7, Q(s, ⬇️) = 7.5, Q(s, ⬅️) = −2. Con ε = 0, ¿qué hace el agente?
- a) ⬆️ · b) ➡️ · c) ⬇️ · d) ⬅️

**3.** ¿Para qué sirve la **memoria de repetición** en DQN?
- a) Para guardar la tabla Q · b) Para entrenar con lotes variados de experiencias pasadas y no solo con la última · c) Para explorar · d) Para bajar γ

**4.** ¿Por qué no se usa una tabla Q para jugar Atari a partir de los píxeles?
- a) Porque las tablas no admiten números negativos · b) Porque hay demasiados estados posibles y la tabla no generaliza entre pantallas parecidas · c) Porque Atari no tiene recompensas · d) Porque ε debe ser 0

**5.** En la regla `objetivo = r + γ · max Q(s')`, si el episodio **terminó** en s'…
- a) Se usa igual · b) Se usa solo `r` (no hay futuro) · c) Se usa solo `max Q(s')` · d) Se reinicia la tabla

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)**
2. **c)** ⬇️ tiene el Q más alto (7.5).
3. **b)** Y además rompe la correlación entre experiencias consecutivas, que desestabiliza el entrenamiento.
4. **b)**
5. **b)** Por eso en el código aparece `0 if terminado else gamma * Q[s2].max()`.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** En el Ejercicio 4 el robot seguía dando ~9 pasos de media al final, aunque ya "sabía" el camino de 8. ¿Por qué? ¿Cómo medirías su **verdadera** habilidad?

**B. Caso de análisis 💼** Un centro de distribución quiere usar Q-learning para que sus robots elijan rutas. Tiene un simulador de la bodega y también un registro de **un año de rutas de los conductores humanos**. ¿Cómo aprovecharías ese registro, sabiendo que Q-learning es *off-policy*? ¿Qué riesgo tiene aprender **solo** de ese registro?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Porque durante el entrenamiento sigue explorando (ε = 0.1). Para medir su habilidad real hay que **evaluarlo aparte con ε = 0**, como hace `seguir_politica`. Es el error común de la Lección 14.2.

**B.** El registro humano se puede meter directamente en la **memoria de repetición**: son experiencias (estado, acción, recompensa, siguiente estado) válidas aunque las haya generado otra política. Así el agente arranca sabiendo algo antes de usar el simulador. El riesgo: los humanos probablemente **nunca** probaron ciertas rutas, así que el agente no sabe cuánto valen. Podría sobreestimarlas (el `max` elige justo los valores mal estimados) y recomendar rutas que nadie verificó. Por eso conviene complementar con el simulador. (Este problema tiene nombre propio: **RL offline**.)
</details>

### 9.3 🧩 Práctica integradora: "¿El atajo junto a la lava?"

Una bodega tiene un pasillo **corto** que pasa justo junto a un foso de lava 🔥 y un pasillo **largo y seguro** por arriba. Caer en la lava es grave: cuesta **−50** (un robot roto). Vamos a ver qué ruta aprende el robot según qué tan resbaloso esté el piso.

```
⬜ ⬜ ⬜ ⬜ ⬜ ⬜      ← ruta larga y segura (11 pasos)
⬜ 🧱 🧱 🧱 🧱 ⬜
⬜ ⬜ ⬜ ⬜ ⬜ ⬜      ← ruta corta, junto a la lava (7 pasos)
🟩 🔥 🔥 🔥 🔥 💎
```

**Paso 1 ▶️ — El mapa**

In [ ]:
BODEGA = ["......",
          ".####.",
          "......",
          "SXXXXG"]
dibujar(BODEGA)

**Paso 2 🔮 — Piso seco.** Entrenamos con `resbalon = 0` (sin resbalones). ¿Qué ruta aprenderá, la corta o la larga?

✍️ *Mi predicción:*

In [ ]:
RESBALON = 0.0          # ✏️ en el Paso 3 cambiarás este valor
CASTIGO_LAVA = -50      # ✏️ en el Paso 4 cambiarás este valor

Q_b, pasos_b = q_learning(Laberinto(BODEGA, resbalon=RESBALON, castigo_lava=CASTIGO_LAVA, semilla=1),
                          episodios=3000, alfa=0.1, gamma=0.99, epsilon=0.1, semilla=1)
camino, final = seguir_politica(BODEGA, Q_b)
print(f"Resbalón = {RESBALON} · lava = {CASTIGO_LAVA} → ruta de {len(camino) - 1} pasos, {final}")
dibujar(BODEGA, camino=camino)

<details><summary><b>👀 Ver respuesta</b></summary>

La **corta (7 pasos)**. Sin resbalones, pasar junto a la lava no tiene ningún riesgo: el robot va exactamente adonde quiere, así que el camino más corto es el mejor.
</details>

**Paso 3 ✏️ — Piso mojado.** Cambia `RESBALON` a **0.1** y vuelve a ejecutar la celda del Paso 2. ¿Cambia la ruta? ¿Por qué?

✍️ *Mi observación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con `RESBALON = 0.1` el robot elige la **ruta larga y segura (11 pasos)**. Cada paso junto a la lava tiene un 2.5 % de probabilidad de resbalar justo hacia abajo (10 % de resbalar × 1/4 de que sea "abajo"). En los 4 pasos sobre el foso, el riesgo acumulado ronda el 10 %, y cada caída cuesta −50: en promedio, unos −5. Caminar 4 pasos extra (−4) sale más barato que ese riesgo.

Lo importante: **nadie le dijo al robot "evita la lava cuando el piso está mojado"**. Lo dedujo solo, porque Q aprende el valor **esperado** de cada acción, incluido el riesgo.
</details>

**Paso 4 ✏️ — ¿Cuánto miedo le tiene a la lava?** Deja `RESBALON = 0.1` y prueba `CASTIGO_LAVA` = **−10** y luego **−1**. ¿Qué ruta elige en cada caso? ¿Qué te dice eso sobre el diseño de la recompensa (Lección 14.1)?

✍️ *Mi observación:*

<details><summary><b>👀 Ver respuesta</b></summary>

- Con **−10** el robot vuelve a la **ruta corta (7 pasos)**: el riesgo (≈ 10 % × −10 ≈ −1, más el premio perdido) ya no compensa caminar 4 pasos más.
- Con **−1** pasa algo inquietante: el robot **se tira directamente a la lava en el primer paso**. Caer cuesta −1, lo mismo que un paso normal, y además termina el episodio. Al principio, cuando el robot aún no conoce el diamante, cualquier paso le cuesta −1 y "seguir caminando" le parece más caro que terminar ya. Como se rinde enseguida, **casi nunca llega al diamante** y nunca descubre que valía +10. (Si lo mira la tabla: Q(salida, ➡️ a la lava) = −1 y las demás acciones valen ≈ −2 o menos.)

Es *reward hacking* (Lección 14.1) combinado con falta de exploración (Lección 14.2): para el agente, **rendirse** es la opción más rentable que conoce.

Moraleja: **la recompensa define qué riesgos acepta el agente.** En un robot real, si "romper algo" no se castiga lo suficiente en la recompensa, el agente aprenderá a correr ese riesgo.
</details>

**Paso 5 ✋ — Conclusión para el jefe de operaciones.** Escribe en 2 o 3 frases qué aprendiste para alguien que no sabe ML: ¿de qué depende la ruta que elige el robot?

✍️ *Mi conclusión:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:

> «El robot elige su ruta comparando el **tiempo** que ahorra con el **riesgo** que corre. Cuando el piso está seco toma el atajo; cuando está mojado prefiere el camino largo, porque una caída cuesta más que unos pasos extra. Pero solo lo hace si le enseñamos (con la recompensa) que caer es grave: **los valores que le damos a cada error deciden qué tan prudente es**, así que esos valores deben definirse junto con el equipo de seguridad.»
</details>

---
## 📚 Para profundizar
- 📘 **Libro:** Sutton y Barto, capítulo 6 (*Temporal-Difference Learning*): Q-learning, SARSA y el famoso ejemplo del acantilado (*cliff walking*), primo de nuestra bodega.
- 📰 **Artículo:** Mnih et al. (2015), *Human-level control through deep reinforcement learning* (Nature): el DQN que jugó 49 juegos de Atari.
- 📺 **Video:** DeepMind, el DQN aprendiendo *Breakout* (busca "DQN Breakout"): descubre solo la estrategia de abrir un túnel por el costado.
- 🧪 **Para Colab:** `stable-baselines3` (DQN, PPO y más listos para usar) y `gymnasium` (entornos).

---
⬅️ **Anterior:** [14.2 Exploración vs. explotación: ¿lo de siempre o algo nuevo?](14.2-exploracion-vs-explotacion.ipynb) · ➡️ **Siguiente:** [14.4 Gradientes de política y PPO: más de lo que funcionó](14.4-gradientes-de-politica-y-ppo.ipynb)